## MODELO PREDICTIVO DE BROTES DE DENGUE POR VARIABLES CLIMATICAS EN EL PERU PARA LA EMPRESA MINSA

Pipeline nacional con arquitectura Bronze, Silver y Gold

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ModeloPredictivoDenguePeru").getOrCreate()

def resolver_ruta_datos(nombre_archivo):
    candidatos = [
        Path.cwd() / "data" / "raw" / nombre_archivo,
        Path.cwd().parent / "data" / "raw" / nombre_archivo,
        Path("/home/jovyan/work/data/raw") / nombre_archivo,
    ]
    for candidato in candidatos:
        if candidato.exists():
            return str(candidato)
    revisadas = "\n".join(f"- {ruta}" for ruta in candidatos)
    raise FileNotFoundError(
        f"No se encontro {nombre_archivo}. Coloca el archivo en data/raw.\n{revisadas}"
    )

def ruta_salida(capa, dataset):
    base = Path.cwd()
    if base.name == "notebooks":
        base = base.parent
    return str(base / "data" / capa / dataset)

In [2]:
from pyspark.sql.functions import col, trim, upper

ruta_dengue = resolver_ruta_datos("datos_abiertos_vigilancia_dengue_2000_2024.csv")
df_dengue = spark.read.csv(ruta_dengue, header=True, inferSchema=True, sep=";")
# Bronze conserva la ingesta original y no se sobrescribe en ejecuciones posteriores.
df_dengue.write.mode("ignore").parquet(ruta_salida("bronze", "dengue"))
columnas_requeridas = {
    "departamento", "provincia", "distrito", "enfermedad",
    "ano", "semana", "edad", "sexo"
}
faltantes = columnas_requeridas.difference(df_dengue.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en el archivo de Dengue: {sorted(faltantes)}")

print(f"Volumen de Dengue: {df_dengue.count()} registros y {len(df_dengue.columns)} columnas")
df_dengue.printSchema()

# Solo Dengue y a nivel nacional; no se filtra ningun departamento.
df_dengue_nacional = (
    df_dengue
    .filter(
        upper(trim(col("enfermedad"))).contains("DENGUE")
    )
    .withColumn("ano", col("ano").cast("int"))
    .withColumn("semana", col("semana").cast("int"))
)
print(f"Total de registros nacionales de Dengue: {df_dengue_nacional.count()}")
df_dengue_nacional.write.mode("overwrite").partitionBy("ano").parquet(
    ruta_salida("silver", "dengue_nacional")
)
print("Dengue guardado en Silver como Parquet particionado por ano")

Volumen de Dengue: 1029421 registros y 14 columnas
root
 |-- departamento: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- enfermedad: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- semana: integer (nullable = true)
 |-- diagnostic: string (nullable = true)
 |-- diresa: integer (nullable = true)
 |-- ubigeo: integer (nullable = true)
 |-- localcod: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- tipo_edad: string (nullable = true)
 |-- sexo: string (nullable = true)

Total de registros nacionales de Dengue: 1029421
Dengue guardado en Silver como Parquet particionado por ano


In [3]:
from pyspark.sql.functions import avg, date_add, lit, make_date, round, sum as spark_sum, weekofyear

ruta_clima = resolver_ruta_datos("datos_climaticos_2000_2024.csv")
df_clima = spark.read.csv(
    ruta_clima,
    header=True,
    inferSchema=True,
    sep=",",
    nullValue="NA"
)
df_clima.write.mode("ignore").parquet(ruta_salida("bronze", "clima"))

columnas_clima = {"YEAR", "DOY", "T2M", "RH2M", "PRECTOTCORR"}
faltantes_clima = columnas_clima.difference(df_clima.columns)
if faltantes_clima:
    raise ValueError(f"Faltan columnas en el archivo climatico: {sorted(faltantes_clima)}")

# Se construye la fecha desde ano y dia del ano sin depender de formatos ambiguos.
df_clima_fechas = (
    df_clima
    .withColumn("YEAR", col("YEAR").cast("int"))
    .withColumn("DOY", col("DOY").cast("int"))
    .withColumn("T2M", col("T2M").cast("double"))
    .withColumn("RH2M", col("RH2M").cast("double"))
    .withColumn("PRECTOTCORR", col("PRECTOTCORR").cast("double"))
    .withColumn("fecha_exacta", date_add(make_date(col("YEAR"), lit(1), lit(1)), col("DOY") - lit(1)))
)
df_clima_semanas = df_clima_fechas.withColumn("semana", weekofyear(col("fecha_exacta")))
df_clima_agrupado = (
    df_clima_semanas.groupBy("YEAR", "semana")
    .agg(
        round(avg("T2M"), 2).alias("temp_promedio"),
        round(avg("RH2M"), 2).alias("humedad_promedio"),
        round(spark_sum("PRECTOTCORR"), 2).alias("precipitacion_total")
    )
    .orderBy("YEAR", "semana")
)
df_clima_agrupado.show(5)
df_clima_agrupado.write.mode("overwrite").partitionBy("YEAR").parquet(
    ruta_salida("silver", "clima_semanal")
)
print("Clima guardado en Silver como Parquet particionado por YEAR")

+----+------+-------------+----------------+-------------------+
|YEAR|semana|temp_promedio|humedad_promedio|precipitacion_total|
+----+------+-------------+----------------+-------------------+
|2000|     1|        15.78|           68.89|               0.49|
|2000|     2|         16.3|           70.75|               1.18|
|2000|     3|        17.04|            70.9|               5.14|
|2000|     4|        16.78|            75.2|               3.66|
|2000|     5|        16.62|           76.61|               7.64|
+----+------+-------------+----------------+-------------------+
only showing top 5 rows

Clima guardado en Silver como Parquet particionado por YEAR


In [4]:
df_dengue_silver = spark.read.parquet(ruta_salida("silver", "dengue_nacional"))
df_clima_silver = (
    spark.read.parquet(ruta_salida("silver", "clima_semanal"))
    .withColumnRenamed("YEAR", "ano")
)

# La tabla climatica debe tener una sola fila por ano y semana.
duplicados_clima = (
    df_clima_silver.groupBy("ano", "semana").count().filter(col("count") > 1).limit(1).count()
)
if duplicados_clima:
    raise ValueError("La tabla climatica tiene varias filas para la misma combinacion ano-semana")

df_gold = df_dengue_silver.join(df_clima_silver, on=["ano", "semana"], how="inner")
total_cruzado = df_gold.count()
print(f"Total de registros tras el cruce nacional (Gold): {total_cruzado}")
df_gold.select(
    "ano", "semana", "departamento", "provincia", "distrito", "edad", "sexo",
    "temp_promedio", "humedad_promedio", "precipitacion_total"
).show(10)

df_gold.write.mode("overwrite").partitionBy("ano").parquet(
    ruta_salida("gold", "dengue_clima_nacional")
)
print("Dataset integrado guardado en Gold como Parquet particionado por ano")

Total de registros tras el cruce nacional (Gold): 1028782
+----+------+------------+---------+--------+----+----+-------------+----------------+-------------------+
| ano|semana|departamento|provincia|distrito|edad|sexo|temp_promedio|humedad_promedio|precipitacion_total|
+----+------+------------+---------+--------+----+----+-------------+----------------+-------------------+
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  29|   M|        18.61|            71.9|               1.26|
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  11|   F|        18.61|            71.9|               1.26|
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  35|   F|        18.61|            71.9|               1.26|
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  39|   F|        18.61|            71.9|               1.26|
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  11|   F|        18.61|            71.9|               1.26|
|2023|    22|  LAMBAYEQUE| CHICLAYO|CHICLAYO|  57|   M|        18.61|            71.9|

La salida Gold queda en `data/gold/dengue_clima_nacional`. Spark crea una carpeta Parquet particionada por `ano`; no es un unico archivo CSV.

In [9]:
df_dengue.select("enfermedad").distinct().show(100, truncate=False)

+---------------------------+
|enfermedad                 |
+---------------------------+
|DENGUE CON SIGNOS DE ALARMA|
|DENGUE SIN SIGNOS DE ALARMA|
|DENGUE GRAVE               |
+---------------------------+



In [10]:
df_dengue.groupBy("enfermedad").count().orderBy("count", ascending=False).show(100, truncate=False)

+---------------------------+------+
|enfermedad                 |count |
+---------------------------+------+
|DENGUE SIN SIGNOS DE ALARMA|915243|
|DENGUE CON SIGNOS DE ALARMA|110166|
|DENGUE GRAVE               |4012  |
+---------------------------+------+

